# day-19-agent-concepts — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [11]:
# ---- Solution 1 ----
@tool("set_budget", "Set a team's monthly budget (DESTRUCTIVE).",
      {"team": {"type": "string"}, "amount": {"type": "number"}}, destructive=True)
def set_budget(team, amount):
    if team not in TEAMS: return {"error": "unknown team"}
    TEAMS[team]["monthly_budget"] = amount
    return {"ok": True, "team": team, "new_budget": amount}

def raise_design_planner(messages):
    n = len([m for m in messages if m["role"] == "tool"])
    if n == 0: return Action("tool", name="set_budget", args={"team": "design", "amount": 200000})
    return Action("final", text="done")

def deny(act):
    print(f"  APPROVAL REQUEST: {act.name}({act.args})  -> DENIED")
    return False

print(run_agent_final("Raise the design budget to 200k", raise_design_planner, on_destructive=deny))
print("design budget unchanged:", TEAMS["design"]["monthly_budget"])

  APPROVAL REQUEST: set_budget({'team': 'design', 'amount': 200000})  -> DENIED
{'answer': 'HALTED: set_budget needs approval', 'steps': 0, 'log': [{'step': 0, 'kind': 'tool', 'detail': 'set_budget'}]}
design budget unchanged: 110000


In [12]:
# ---- Solution 6 ----
def steps_to_exceed(limit):
    s = 1
    while agent_cost(s) < limit: s += 1
    return s
print(f"S6: exceeds $0.10 at {steps_to_exceed(0.10)} steps; exceeds $1.00 at {steps_to_exceed(1.00)} steps")
print("    -> default max_steps in the 6-12 range for most agents; only raise it with a hard")
print("       token/dollar budget guard also in place.")

S6: exceeds $0.10 at 13 steps; exceeds $1.00 at 47 steps
    -> default max_steps in the 6-12 range for most agents; only raise it with a hard
       token/dollar budget guard also in place.


### Solutions 2, 3, 4, 5 (sketch)

**S2:** keep a `deque(maxlen=3)` of `json.dumps(result)`; if `len(set(...)) == 1` and it's
full, abort with "no progress". Catches a model that varies its query but keeps getting the
same nothing back.

**S3:** planner steps: `get_team("design")` → `get_finance("cash_on_hand")` →
`get_finance("monthly_revenue")` → `calculator("6400000 / ((110000*2 - 110000) + (300000 -
... ))")` ... then `final`: "Doubling design adds $110k/month; net burn becomes $440k/month;
$6.4M / $440k ≈ 14.5 months — yes, affordable for a year."

**S4:** `enc = tiktoken.get_encoding("cl100k_base")`; before each planner call,
`if len(enc.encode(json.dumps(messages))) + est_response > max_input_tokens: abort`. Agents
die from context growth, not from step count alone.

**S5:** the recovery planner checks the last tool result: `if last and "error" in last:` pick
a different valid argument. A robust agent treats tool errors as observations to reason about,
not as crashes.

### Answer key
1. The model chooses the control flow — which action to take next and when to stop — instead
   of following a fixed pipeline you wrote.
2. The tool's result (the "observation"), appended to the conversation so the next model call
   can reason about it.
3. Infinite/repeated loop → repeated-action detection + `max_steps`; wrong or destructive
   action → tool-side validation + tool allowlist + human approval; runaway cost → tool-call
   budget + token-budget guard. (Also: hanging tool → per-call timeout.)
4. The model's tool arguments are untrusted input — it can hallucinate bad values or be
   prompt-injected. A prompt instruction is a suggestion; validation in the function is
   enforcement.
5. Each step is a fresh LLM call whose input is the entire growing transcript, so per-step
   input tokens rise roughly linearly with step number, making cumulative cost roughly
   quadratic.
6. When the action is destructive or irreversible and hard to recover from: deleting data,
   sending communications, spending money, deploying, modifying production.
7. One message, role `user`, containing all three `tool_result` content blocks. Splitting them
   across messages trains the model to stop making parallel calls.